# CrewAI + Ollama + Guardrails AI
## Video Compliance & Violation Detection POC



## Step 1 - Install Required Libraries


In [1]:
# uv pip install -r requirements.txt

## Step 2 - Import Libraries


In [ ]:
import os
import json

from pydantic import BaseModel
from typing import Literal

from crewai import Agent, Task, Crew, Process, LLM
from guardrails import Guard

print("Libraries imported successfully.")


Libraries imported successfully.


## Step 4 - Configure ollama LLM


In [3]:
from crewai import LLM

llm = LLM(
    model="ollama/llama3.2",
    base_url="http://localhost:11434",
    temperature=0
)

print("CrewAI Ollama LLM configured")

CrewAI Ollama LLM configured


## Step 5 - Simulated Video Content


In [4]:
video_content = """
VIDEO TRANSCRIPT:

This weight loss product helps you lose 10 kg
in only 10 days without exercise.

You will get guaranteed results with our product.

ON-SCREEN TEXT:

Lose 10 KG in 10 Days
100% Guaranteed Results
Best weight loss solution.
"""

print(video_content)



VIDEO TRANSCRIPT:

This weight loss product helps you lose 10 kg
in only 10 days without exercise.

You will get guaranteed results with our product.

ON-SCREEN TEXT:

Lose 10 KG in 10 Days
100% Guaranteed Results
Best weight loss solution.



## Step 6 - Compliance Rules


In [5]:
compliance_rules = """
RULE 1:
Advertisements must not make unsupported health claims.

RULE 2:
Advertisements must not guarantee specific results
without appropriate evidence.

RULE 3:
Advertisements must not make misleading or unrealistic
weight-loss claims.

RULE 4:
Health-related claims should be supported by appropriate evidence.
"""

print(compliance_rules)



RULE 1:
Advertisements must not make unsupported health claims.

RULE 2:
Advertisements must not guarantee specific results
without appropriate evidence.

RULE 3:
Advertisements must not make misleading or unrealistic
weight-loss claims.

RULE 4:
Health-related claims should be supported by appropriate evidence.



## Step 7 - Create Compliance Analyst Agent


In [6]:
compliance_analyst = Agent(
    role="Advertisement Compliance Analyst",
    goal=(
        "Analyze advertisement video content and identify "
        "claims, promotional statements, and potential compliance risks."
    ),
    backstory=(
        "You are an experienced advertisement compliance analyst. "
        "Analyze transcript and on-screen text. Identify health claims, "
        "guaranteed results, misleading statements, and other risks. "
        "Do not make the final compliance decision."
    ),
    llm=llm,
    verbose=True
)

print("Compliance Analyst created.")


Compliance Analyst created.


## Step 8 - Create Senior Compliance Reviewer Agent


In [7]:
compliance_reviewer = Agent(
    role="Senior Compliance Reviewer",
    goal=(
        "Review the analyst findings against the compliance rules "
        "and produce the final compliance decision."
    ),
    backstory=(
        "You are a senior advertisement compliance reviewer. "
        "Compare claims with the supplied rules. "
        "Return a clear status and explain every violation."
    ),
    llm=llm,
    verbose=True
)

print("Compliance Reviewer created.")


Compliance Reviewer created.


## Step 9 - Create Analyst Task


In [8]:
analysis_task = Task(
    description=f"""
Analyze the following advertisement content:

{video_content}

Identify:
1. Main claims
2. Health or performance claims
3. Guaranteed-result claims
4. Potentially misleading statements
5. Compliance concerns

Do not give the final PASS or FAIL decision.
Provide clear findings for the senior reviewer.
""",
    expected_output="A clear list of advertisement claims and potential compliance concerns.",
    agent=compliance_analyst
)

print("Analyst task created.")


Analyst task created.


## Step 10 - Create Review Task


In [9]:
review_task = Task(
    description=f"""
Review the advertisement analysis produced by the previous agent.

Use these compliance rules:

{compliance_rules}

Create a final compliance decision.

Return:
- STATUS: PASS, FAIL, or REQUIRES_REVIEW
- COMPLIANCE RESULTS: category, severity, description
- FINAL REPORT: short explanation

Be factual and base the decision only on the supplied content and rules.
""",
    expected_output="A final compliance decision with violations and a short report.",
    agent=compliance_reviewer,
    context=[analysis_task]
)

print("Review task created.")


Review task created.


## Step 11 - Create CrewAI Crew


In [10]:
compliance_crew = Crew(
    agents=[compliance_analyst, compliance_reviewer],
    tasks=[analysis_task, review_task],
    process=Process.sequential,
    verbose=True
)

print("Crew created successfully.")


Crew created successfully.


## Step 12 - Run CrewAI


In [11]:
result = await compliance_crew.kickoff_async()
print("Crew execution completed.")

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c71c296e-3517-4c54-81d0-1265d8d53902                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  Analyze the following advertisement content:                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  VIDEO TRANSCRIPT:                                                                                              │
│                                                                                                                 │
│  This weight loss product helps you lose 10 kg                                                                  │
│  in only 10 days without exercise.                                                                              │
│                                                                                                                 │
│  You will get guaranteed results with our product.                                                              │
│                                                                                                                 │
│  ON-SCREEN TEXT:                                                                                                │
│                                                                                                                 │
│  Lose 10 KG in 10 Days                                                                                          │
│  100% Guaranteed Results                                                                                        │
│  Best weight loss solution.                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│  Identify:                                                                                                      │
│  1. Main claims                                                                                                 │
│  2. Health or performance claims                                                                                │
│  3. Guaranteed-result claims                                                                                    │
│  4. Potentially misleading statements                                                                           │
│  5. Compliance concerns                                                                                         │
│                                                                                                                 │
│  Do not give the final PASS or FAIL decision.                                                                   │
│  Provide clear findings for the senior reviewer.                                                                │
│                                                                                                                 │
│  ID: f9a4c8e2-cf48-4fa1-885a-66c99c64b5a9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Advertisement Compliance Analyst                                                                        │
│                                                                                                                 │
│  Task:                                                                                                          │
│  Analyze the following advertisement content:                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  VIDEO TRANSCRIPT:                                                                                              │
│                                                                                                                 │
│  This weight loss product helps you lose 10 kg                                                                  │
│  in only 10 days without exercise.                                                                              │
│                                                                                                                 │
│  You will get guaranteed results with our product.                                                              │
│                                                                                                                 │
│  ON-SCREEN TEXT:                                                                                                │
│                                                                                                                 │
│  Lose 10 KG in 10 Days                                                                                          │
│  100% Guaranteed Results                                                                                        │
│  Best weight loss solution.                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│  Identify:                                                                                                      │
│  1. Main claims                                                                                                 │
│  2. Health or performance claims                                                                                │
│  3. Guaranteed-result claims                                                                                    │
│  4. Potentially misleading statements                                                                           │
│  5. Compliance concerns                                                                                         │
│                                                                                                                 │
│  Do not give the final PASS or FAIL decision.                                                                   │
│  Provide clear findings for the senior reviewer.                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Advertisement Compliance Analyst                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Advertisement Analysis Report**                                                                              │
│                                                                                                                 │
│  **VIDEO TRANSCRIPT:**                                                                                          │
│                                                                                                                 │
│  1. Main claims:                                                                                                │
│          * "This weight loss product helps you lose 10 kg in only 10 days without exercise."                    │
│  2. Health or performance claims:                                                                               │
│          * None explicitly stated, but implies a health-related benefit (weight loss).                          │
│  3. Guaranteed-result claims:                                                                                   │
│          * "You will get guaranteed results with our product."                                                  │
│  4. Potentially misleading statements:                                                                          │
│          * The claim that the product can help lose 10 kg in 10 days without exercise may be unrealistic and    │
│  potentially misleading.                                                                                        │
│  5. Compliance concerns:                                                                                        │
│          * The advertisement does not provide any scientific evidence or credible sources to support its        │
│  claims, which raises concerns about its compliance with advertising regulations.                               │
│                                                                                                                 │
│  **ON-SCREEN TEXT:**                                                                                            │
│                                                                                                                 │
│  1. Main claims:                                                                                                │
│          * "Lose 10 KG in 10 Days"                                                                              │
│          * "100% Guaranteed Results"                                                                            │
│          * "Best weight loss solution."                                                                         │
│  2. Health or performance claims:                                                                               │
│          * None explicitly stated, but implies a health-related benefit (weight loss).                          │
│  3. Guaranteed-result claims:                                                                                   │
│          * "100% Guaranteed Results"                                                                            │
│  4. Potentially misleading statements:                                                                          │
│          * The use of the phrase "100% Guaranteed Results" may be overly promotional and potentially            │
│  misleading.                                           

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Compliance Reviewer                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **FINAL COMPLIANCE DECISION**                                                                                  │
│                                                                                                                 │
│  **STATUS:** FAIL                                                                                               │
│                                                                                                                 │
│  **COMPLIANCE RESULTS:**                                                                                        │
│                                                                                                                 │
│  | RULE | SEVERITY | DESCRIPTION |                                                                              │
│  | --- | --- | --- |                                                                                            │
│  | RULE 1: Advertisements must not make unsupported health claims. | HIGH | The advertisement makes a claim     │
│  that the product can help lose 10 kg in 10 days without exercise, which is an unsupported health claim. |      │
│  | RULE 2: Advertisements must not guarantee specific results without appropriate evidence. | HIGH | The        │
│  advertisement guarantees "100% Guaranteed Results" without providing any scientific evidence or credible       │
│  sources to support its claims. |                                                                               │
│  | RULE 3: Advertisements must not make misleading or unrealistic weight-loss claims. | HIGH | The claim that   │
│  the product can help lose 10 kg in 10 days without exercise may be unrealistic and potentially misleading. |   │
│  | RULE 4: Health-related claims should be supported by appropriate evidence. | HIGH | The advertisement does   │
│  not provide any scientific evidence or credible sources to support its health-related claims, which raises     │
│  concerns about its compliance with advertising regulations. |                                                  │
│                                                                                                                 │
│  **FINAL REPORT:**                                                                                              │
│                                                                                                                 │
│  The advertisement analysis report reveals several serious compliance issues with relevant advertising          │
│  regulations. The advertisement makes unsupported health claims, guarantees specific results without evidence,  │
│  and makes misleading weight-loss claims. Furthermore, the advertisement does not provide any credible sources  │
│  or references to support its claims, which raises concerns about its safety and efficacy. The language used    │
│  is highly promotional and exaggerated, which may be seen as deceptive. In light of these findings, I conclude  │
│  that the advertisement fails to comply with relevant advertising regulations.                                  │
│                                                                                                                 │
│  **RECOMMENDATIONS:**                                                                                           │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  Review the advertisement analysis produced by the previous agent.                                              │
│                                                                                                                 │
│  Use these compliance rules:                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
│  RULE 1:                                                                                                        │
│  Advertisements must not make unsupported health claims.                                                        │
│                                                                                                                 │
│  RULE 2:                                                                                                        │
│  Advertisements must not guarantee specific results                                                             │
│  without appropriate evidence.                                                                                  │
│                                                                                                                 │
│  RULE 3:                                                                                                        │
│  Advertisements must not make misleading or unrealistic                                                         │
│  weight-loss claims.                                                                                            │
│                                                                                                                 │
│  RULE 4:                                                                                                        │
│  Health-related claims should be supported by appropriate evidence.                                             │
│                                                                                                                 │
│                                                                                                                 │
│  Create a final compliance decision.                                                                            │
│                                                                                                                 │
│  Return:                                                                                                        │
│  - STATUS: PASS, FAIL, or REQUIRES_REVIEW                                                                       │
│  - COMPLIANCE RESULTS: category, severity, description                                                          │
│  - FINAL REPORT: short explanation                                                                              │
│                                                                                                                 │
│  Be factual and base the decision only on the supplied content and rules.                                       │
│                                                                                                                 │
│  Agent: Senior Compliance Reviewer                                                                              │
│                                                        

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c71c296e-3517-4c54-81d0-1265d8d53902                                                                       │
│  Final Output: **FINAL COMPLIANCE DECISION**                                                                    │
│                                                                                                                 │
│  **STATUS:** FAIL                                                                                               │
│                                                                                                                 │
│  **COMPLIANCE RESULTS:**                                                                                        │
│                                                                                                                 │
│  | RULE | SEVERITY | DESCRIPTION |                                                                              │
│  | --- | --- | --- |                                                                                            │
│  | RULE 1: Advertisements must not make unsupported health claims. | HIGH | The advertisement makes a claim     │
│  that the product can help lose 10 kg in 10 days without exercise, which is an unsupported health claim. |      │
│  | RULE 2: Advertisements must not guarantee specific results without appropriate evidence. | HIGH | The        │
│  advertisement guarantees "100% Guaranteed Results" without providing any scientific evidence or credible       │
│  sources to support its claims. |                                                                               │
│  | RULE 3: Advertisements must not make misleading or unrealistic weight-loss claims. | HIGH | The claim that   │
│  the product can help lose 10 kg in 10 days without exercise may be unrealistic and potentially misleading. |   │
│  | RULE 4: Health-related claims should be supported by appropriate evidence. | HIGH | The advertisement does   │
│  not provide any scientific evidence or credible sources to support its health-related claims, which raises     │
│  concerns about its compliance with advertising regulations. |                                                  │
│                                                                                                                 │
│  **FINAL REPORT:**                                                                                              │
│                                                                                                                 │
│  The advertisement analysis report reveals several serious compliance issues with relevant advertising          │
│  regulations. The advertisement makes unsupported health claims, guarantees specific results without evidence,  │
│  and makes misleading weight-loss claims. Furthermore, the advertisement does not provide any credible sources  │
│  or references to support its claims, which raises concerns about its safety and efficacy. The language used    │
│  is highly promotional and exaggerated, which may be seen as deceptive. In light of these findings, I conclude  │
│  that the advertisement fails to comply with relevant advertising regulations.                                  │
│                                                                                                                 │
│  **RECOMMENDATIONS:**                                                                                           │
│                                                       

Crew execution completed.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Step 13 - Define Guardrails Output Schema


In [ ]:

class ComplianceResult(BaseModel):
    category: str
    severity: str
    description: str


class FinalComplianceReport(BaseModel):
    status: Literal["PASS", "FAIL", "REQUIRES_REVIEW"]
    compliance_results: list[ComplianceResult]
    final_report: str

## Step 14 - Prepare Guardrails


In [13]:
guard = Guard.for_pydantic(output_class=FinalComplianceReport)

raw_output = result.raw

print("Raw CrewAI output:")
print(raw_output)


Raw CrewAI output:
**FINAL COMPLIANCE DECISION**

**STATUS:** FAIL

**COMPLIANCE RESULTS:**

| RULE | SEVERITY | DESCRIPTION |
| --- | --- | --- |
| RULE 1: Advertisements must not make unsupported health claims. | HIGH | The advertisement makes a claim that the product can help lose 10 kg in 10 days without exercise, which is an unsupported health claim. |
| RULE 2: Advertisements must not guarantee specific results without appropriate evidence. | HIGH | The advertisement guarantees "100% Guaranteed Results" without providing any scientific evidence or credible sources to support its claims. |
| RULE 3: Advertisements must not make misleading or unrealistic weight-loss claims. | HIGH | The claim that the product can help lose 10 kg in 10 days without exercise may be unrealistic and potentially misleading. |
| RULE 4: Health-related claims should be supported by appropriate evidence. | HIGH | The advertisement does not provide any scientific evidence or credible sources to support its 

## Step 15 - Ask Groq for Structured JSON

The CrewAI result may contain normal text. This step converts the reviewer result into the exact JSON structure required by Guardrails.


In [14]:
json_prompt = f"""
Convert the following compliance review into ONLY valid JSON.

Required structure:
{{
  "status": "PASS or FAIL or REQUIRES_REVIEW",
  "compliance_results": [
    {{
      "category": "string",
      "severity": "string",
      "description": "string"
    }}
  ],
  "final_report": "string"
}}

Compliance review:
{raw_output}
"""

response = llm.call(json_prompt)
print(response)


Here is the compliance review converted into only valid JSON:

```
{
  "status": "FAIL",
  "compliance_results": [
    {
      "category": "Advertisements must not make unsupported health claims.",
      "severity": "HIGH",
      "description": "The advertisement makes a claim that the product can help lose 10 kg in 10 days without exercise, which is an unsupported health claim."
    },
    {
      "category": "Advertisements must not guarantee specific results without appropriate evidence.",
      "severity": "HIGH",
      "description": "The advertisement guarantees \"100% Guaranteed Results\" without providing any scientific evidence or credible sources to support its claims."
    },
    {
      "category": "Advertisements must not make misleading or unrealistic weight-loss claims.",
      "severity": "HIGH",
      "description": "The claim that the product can help lose 10 kg in 10 days without exercise may be unrealistic and potentially misleading."
    },
    {
      "category": 

## Step 16 - Validate Final Output with Guardrails


In [15]:
validated = guard.parse(response)

# Guardrails returns the validated result here.
final_result = getattr(validated, "validated_output", None)
if final_result is None:
    final_result = getattr(validated, "validated_data", None)

print("Guardrails validation successful.")
print(json.dumps(final_result, indent=2))


Guardrails validation successful.
{
  "status": "FAIL",
  "compliance_results": [
    {
      "category": "Advertisements must not make unsupported health claims.",
      "severity": "HIGH",
      "description": "The advertisement makes a claim that the product can help lose 10 kg in 10 days without exercise, which is an unsupported health claim."
    },
    {
      "category": "Advertisements must not guarantee specific results without appropriate evidence.",
      "severity": "HIGH",
      "description": "The advertisement guarantees \"100% Guaranteed Results\" without providing any scientific evidence or credible sources to support its claims."
    },
    {
      "category": "Advertisements must not make misleading or unrealistic weight-loss claims.",
      "severity": "HIGH",
      "description": "The claim that the product can help lose 10 kg in 10 days without exercise may be unrealistic and potentially misleading."
    },
    {
      "category": "Health-related claims should be 

c:\Users\khanf\Downloads\CrewAI_Video_Compliance_Groq_Guardrails_POC_Notebook\.venv\Lib\site-packages\guardrails\validator_service\__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


## Step 17 - Final Compliance Result


In [16]:
print("=" * 70)
print("FINAL COMPLIANCE RESULT")
print("=" * 70)

print("STATUS:", final_result["status"])
print()

for item in final_result["compliance_results"]:
    print("CATEGORY:", item["category"])
    print("SEVERITY:", item["severity"])
    print("DESCRIPTION:", item["description"])
    print()

print("FINAL REPORT:")
print(final_result["final_report"])


FINAL COMPLIANCE RESULT
STATUS: FAIL

CATEGORY: Advertisements must not make unsupported health claims.
SEVERITY: HIGH
DESCRIPTION: The advertisement makes a claim that the product can help lose 10 kg in 10 days without exercise, which is an unsupported health claim.

CATEGORY: Advertisements must not guarantee specific results without appropriate evidence.
SEVERITY: HIGH
DESCRIPTION: The advertisement guarantees "100% Guaranteed Results" without providing any scientific evidence or credible sources to support its claims.

CATEGORY: Advertisements must not make misleading or unrealistic weight-loss claims.
SEVERITY: HIGH
DESCRIPTION: The claim that the product can help lose 10 kg in 10 days without exercise may be unrealistic and potentially misleading.

CATEGORY: Health-related claims should be supported by appropriate evidence.
SEVERITY: HIGH
DESCRIPTION: The advertisement does not provide any scientific evidence or credible sources to support its health-related claims, which raises 

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export spans batch due to timeout, max retries or shutdown.


## POC Architecture

```text
Video Transcript + OCR
          ↓
CrewAI Compliance Analyst
          ↓
Advertisement Analysis
          ↓
CrewAI Senior Reviewer
          ↓
Groq Llama Model
          ↓
Structured JSON
          ↓
Guardrails AI
          ↓
Validated Compliance Report
          ↓
PASS / FAIL / REQUIRES_REVIEW
```

### Relation to the main project

The production project uses Azure Video Indexer, Azure AI Search, Azure OpenAI, LangGraph, LangSmith and Guardrails.

This notebook is only a small CrewAI proof of concept. It demonstrates the same compliance idea using two role-based CrewAI agents, Groq, and Guardrails validation.
